In [1]:
import json
import torch
import torch.nn as nn
import math
from huggingface_hub import login, hf_hub_download
from datasets import Dataset
from transformers import PreTrainedTokenizerFast, TrainingArguments, Trainer, DataCollatorForTokenClassification
from safetensors.torch import load_file


In [2]:
!pip install --upgrade torchao peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 35.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 23.4 MB/s eta 0:00:0000:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: peft
    Found existing installation: peft 0.20.0
    Uninstalling peft-0.20.0:
      Successfully uninstalled peft-0.20.0


In [ ]:
login(token=os.environ["HF_TOKEN"])

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_TOKENIZER_REPO= "A-240242/legal-modern-decoder-checkpoints"

In [4]:
def load_phase2_file(filename):
    path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=f"Phase_2_LNER/Train/{filename}", repo_type="dataset", token=HF_TOKEN)
    if filename.endswith(".txt"):
        with open(path, "r", encoding="utf-8") as f:
            return [line.strip() for line in f.readlines() if line.strip()]
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("Downloading Phase 2 Data...")
# Load and merge text dictionaries
text_data = load_phase2_file("fold1-text.json")
text_data.update(load_phase2_file("fold2-text.json"))

# Load and merge gold annotation dictionaries
gold_data = load_phase2_file("fold1-gold.json")
gold_data.update(load_phase2_file("fold2-gold.json"))

entity_types = load_phase2_file("labels.txt")
print(f"Loaded {len(text_data)} total documents across both folds.")

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

Loaded 70 total documents across both folds.


In [5]:
# 1. Build BIO Schema
LABEL_LIST = ["O"]
for entity in entity_types:
    LABEL_LIST.extend([f"B-{entity}", f"I-{entity}"])

LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
NUM_LABELS = len(LABEL_LIST)

# 2. Load Phase 1 Tokenizer
tokenizer = PreTrainedTokenizerFast.from_pretrained(HF_TOKENIZER_REPO, token=HF_TOKEN)
tokenizer.padding_side = "right"

# 3. Manual Sliding Window Alignment
def align_spans_to_tokens(raw_text, gold_spans, tokenizer, max_length=1024, stride=256):
    # Tokenize the ENTIRE document without truncation
    tokenized_full = tokenizer(raw_text, return_offsets_mapping=True, truncation=False)
    full_ids = tokenized_full["input_ids"]
    full_offsets = tokenized_full["offset_mapping"]
    
    chunked_data = {"input_ids": [], "attention_mask": [], "labels": []}
    step = max_length - stride
    
    # Manually slide a window across the massive token arrays
    for i in range(0, len(full_ids), step):
        chunk_ids = full_ids[i : i + max_length]
        chunk_offs = full_offsets[i : i + max_length]
        
        pad_len = max_length - len(chunk_ids)
        chunk_ids = chunk_ids + [tokenizer.pad_token_id] * pad_len
        chunk_offs = chunk_offs + [(0, 0)] * pad_len
        chunk_mask = [1] * (max_length - pad_len) + [0] * pad_len
        
        chunk_labels = []
        current_entity_id = None
        
        for tok_start, tok_end in chunk_offs:
            if tok_start == tok_end:
                chunk_labels.append(-100)
                continue

            sub_text = raw_text[tok_start:tok_end]
            clean_tok_start = tok_start + (len(sub_text) - len(sub_text.lstrip()))
            clean_tok_end = tok_end - (len(sub_text) - len(sub_text.rstrip()))

            matched_label, matched_span_idx = "O", None
            
            for s_idx, span in enumerate(gold_spans):
                g_start, g_end, g_label = span[:3] 
                if not (clean_tok_end <= g_start or clean_tok_start >= g_end):
                    matched_label = g_label
                    matched_span_idx = s_idx
                    break

            if matched_label == "O":
                chunk_labels.append(LABEL2ID["O"])
                current_entity_id = None
            else:
                if current_entity_id == matched_span_idx:
                    chunk_labels.append(LABEL2ID[f"I-{matched_label}"])
                else:
                    chunk_labels.append(LABEL2ID[f"B-{matched_label}"])
                    current_entity_id = matched_span_idx
                    
        chunked_data["input_ids"].append(chunk_ids)
        chunked_data["attention_mask"].append(chunk_mask)
        chunked_data["labels"].append(chunk_labels)

    return chunked_data

tokenizer_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

In [6]:
print("Processing and Aligning Training Spans...")
all_features = {"input_ids": [], "attention_mask": [], "labels": []}

for doc_id, raw_text in text_data.items():
    gold_spans = gold_data.get(doc_id, [])
    aligned = align_spans_to_tokens(raw_text, gold_spans, tokenizer)
    
    all_features["input_ids"].extend(aligned["input_ids"])
    all_features["attention_mask"].extend(aligned["attention_mask"])
    all_features["labels"].extend(aligned["labels"])

train_dataset = Dataset.from_dict(all_features)
print(f"Successfully chunked into {len(train_dataset)} training sequences.")

Processing and Aligning Training Spans...
Successfully chunked into 636 training sequences.


In [7]:
import math
import torch
import torch.nn as nn
from transformers import PretrainedConfig
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download
from peft import LoraConfig, get_peft_model
from transformers.modeling_outputs import TokenClassifierOutput

class HybridEncoderConfig(PretrainedConfig):
    model_type = "hybrid_encoder_ner"
    def __init__(
        self, vocab_size=32000, d_model=768, num_layers=6, 
        num_heads=12, d_ff=2048, pad_token_id=0, eos_token_id=0, **kwargs
    ):
        super().__init__(pad_token_id=pad_token_id, eos_token_id=eos_token_id, **kwargs)
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.d_ff = d_ff

class HybridEncoderForTokenClassification(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embedding = nn.Embedding(config.vocab_size, config.d_model, padding_idx=config.pad_token_id)
        self.pos_embedding = nn.Embedding(2048, config.d_model)
        
        layer = nn.TransformerEncoderLayer(
            d_model=config.d_model, nhead=config.num_heads, 
            dim_feedforward=config.d_ff, dropout=0.1, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=config.num_layers)
        
        # Dedicated Classification Head for 25 BIO tags
        self.classifier = nn.Linear(config.d_model, NUM_LABELS)

    def forward(self, input_ids, labels=None, **kwargs):
        B, S = input_ids.size()
        device = input_ids.device
        
        # 1. Standard Padding Mask (True = ignore token)
        padding_mask = (input_ids == self.config.pad_token_id)
        
        # 2. Embeddings
        positions = torch.arange(S, device=device).unsqueeze(0).expand(B, -1)
        emb = self.embedding(input_ids) * math.sqrt(self.config.d_model) + self.pos_embedding(positions)
        
        # 3. Pure Bidirectional Encoding (No causal prefix masks)
        memory = self.encoder(emb, src_key_padding_mask=padding_mask)
        
        # 4. BIO Tag Logits
        logits = self.classifier(memory)
        
        # 5. Entity-Weighted Loss
        loss = None
        if labels is not None:
            class_weights = torch.ones(logits.size(-1), device=device) * 3.0
            class_weights[0] = 1.0  
            loss_fn = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)
            loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
            
        return TokenClassifierOutput(loss=loss, logits=logits)

print("Initializing Hybrid Encoder Architecture for NER...")
config = HybridEncoderConfig(
    vocab_size=len(tokenizer), 
    pad_token_id=tokenizer.pad_token_id, 
    num_layers=6 # Matching your Phase 1 scale
)
model = HybridEncoderForTokenClassification(config)

print("Downloading and Loading Phase 1 Weights...")
HF_MODEL_REPO = "A-240242/legal-hybridencoder-checkpoints"
model_weight_path = hf_hub_download(repo_id=HF_MODEL_REPO, filename="model.safetensors", token=HF_TOKEN)

# strict=False allows it to cleanly drop the old lm_head and initialize the new classifier
model.load_state_dict(load_file(model_weight_path), strict=False)

# --- HUGGING FACE PEFT LoRA ---
lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    # Targeting the standard PyTorch MLPs and output projections
    target_modules=["linear1", "linear2", "out_proj"], 
    lora_dropout=0.05,
    bias="none",
    modules_to_save=["classifier"] 
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Initializing Hybrid Encoder Architecture for NER...


model.safetensors: reconstructing file:   0%|          |  0.00B /  335MB            

model.safetensors: downloading bytes:           |  0.00B            

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


trainable params: 1,395,481 || all params: 60,647,474 || trainable%: 2.3010


In [8]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./phase2_gqa_lorma",
    report_to="none",
    num_train_epochs=15,          # Give it time to adapt to bidirectional context
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=4e-4,          
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print("Commencing LORMA adaptation for L-NER...")
trainer.train()

Commencing LORMA adaptation for L-NER...


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
50,12.169431
100,6.418901
150,5.338783
200,4.833312
250,4.548794
300,4.344580
350,4.165549
400,3.993447
450,3.974737
500,3.903249


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

TrainOutput(global_step=600, training_loss=5.1110595703125, metrics={'train_runtime': 1158.721, 'train_samples_per_second': 8.233, 'train_steps_per_second': 0.518, 'total_flos': 0.0, 'train_loss': 5.1110595703125, 'epoch': 15.0})

In [9]:
# 4. Inference Decoding Function with Relaxed Span Merger
def decode_token_predictions_to_spans(raw_text, chunks_preds, chunks_offsets):
    extracted_spans = []

    for preds, offsets in zip(chunks_preds, chunks_offsets):
        current_entity = None 

        for pred_id, (tok_start, tok_end) in zip(preds, offsets):
            if tok_start == tok_end:
                continue

            tag = ID2LABEL.get(pred_id, "O")

            if tag.startswith("B-"):
                if current_entity is not None:
                    extracted_spans.append(current_entity)
                current_entity = {"label": tag[2:], "start": tok_start, "end": tok_end}

            elif tag.startswith("I-"):
                label_type = tag[2:]
                if current_entity is not None and current_entity["label"] == label_type:
                    current_entity["end"] = tok_end
                else:
                    if current_entity is not None:
                        extracted_spans.append(current_entity)
                    current_entity = {"label": label_type, "start": tok_start, "end": tok_end}
            else: 
                if current_entity is not None:
                    extracted_spans.append(current_entity)
                    current_entity = None

        if current_entity is not None:
            extracted_spans.append(current_entity)

   # --- THE FIX: Smart Gap Merger ---
    merged_spans = []
    for span in extracted_spans:
        if not merged_spans:
            merged_spans.append(span)
            continue
        
        prev = merged_spans[-1]
        gap = span["start"] - prev["end"]
        
        if span["label"] == prev["label"] and 0 <= gap <= 12:
            # Extract the exact text between the two fragments
            gap_text = raw_text[prev["end"]:span["start"]].lower().strip()
            
            # 1. Is the gap purely whitespace, punctuation, OR numbers (to catch missed months like ".10.")?
            is_safe_gap = all(c in ".,/- &0123456789" for c in gap_text)
            
            # 2. Is the gap a common legal connector?
            is_connector = gap_text in ["", "of", "v", "v.", "vs", "vs.", "and", "&"]
            
            if is_safe_gap or is_connector:
                prev["end"] = span["end"] 
                continue # Safely stitched!
            
        merged_spans.append(span)

    # Clean, Snap Boundaries, and Deduplicate
    final_spans = []
    seen = set()

    for span in merged_spans:
        s, e, label = span["start"], span["end"], span["label"]
        
        clean_start = s
        clean_end = e
        
        while clean_start < clean_end and (raw_text[clean_start].isspace() or raw_text[clean_start] in ".,;:-"):
            clean_start += 1
        while clean_end > clean_start and (raw_text[clean_end-1].isspace() or raw_text[clean_end-1] in ".,;:-"):
            clean_end -= 1

        if clean_start >= clean_end:
            continue

        span_tuple = (clean_start, clean_end, label)
        if span_tuple not in seen:
            seen.add(span_tuple)
            final_spans.append([clean_start, clean_end, label])

    final_spans.sort(key=lambda x: (x[0], x[1]))
    return final_spans

In [10]:
# # --- Verification Test ---
# test_doc = "IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 1128 OF 2026 RAMESH ... APPELLANT"
# test_gold = [
#     [7, 29, "COURT"],      # "SUPREME COURT OF INDIA"
#     [62, 98, "CASENO"],     # "CRIMINAL APPEAL NO. 1128 OF 2026"
#     [99, 105, "APP"]        # "RAMESH"
# ]

# # 1. Encode
# features = align_spans_to_tokens(test_doc, test_gold, tokenizer, max_length=128, stride=32)

# # 2. Simulate perfect model predictions by re-using gold label targets
# simulated_preds = [
#     [lbl if lbl != -100 else 0 for lbl in chunk_lbls] 
#     for chunk_lbls in features["labels"]
# ]

# # 3. Decode
# recovered_spans = decode_token_predictions_to_spans(
#     test_doc, 
#     features["input_ids"], 
#     features["offset_mapping"], 
#     simulated_preds
# )

# # 4. Assert Exact Character-Level Match
# print("Original Gold Spans: ", test_gold)
# print("Recovered Spans:     ", recovered_spans)

# assert recovered_spans == test_gold, "Alignment failed! Offsets do not match exactly."
# print("\nAlignment pipeline verified: 100% character-offset correctness achieved.")

In [11]:
# --- Cell 6: Bulletproof Inference & CodaBench Submission ---
import zipfile
import json
import torch
import gc
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
import os

print("Loading Test Data...")
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO, 
    filename="Phase_2_LNER/Test/fold3-text.json", 
    repo_type="dataset", 
    token=HF_TOKEN
)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Successfully loaded {len(test_data)} documents from the test fold.")

# --- THE FIX: Rebuild ID2LABEL locally in memory ---
LABEL_LIST = ["O"]
for entity in entity_types:
    LABEL_LIST.extend([f"B-{entity}", f"I-{entity}"])
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}

model.eval()
model.to("cuda")
predictions = {}

for doc_id, raw_text in tqdm(test_data.items(), desc="Generating L-NER Spans"):
    torch.cuda.empty_cache()
    gc.collect()
    
    try:
        # 1. Tokenize document without truncation
        tokenized = tokenizer(raw_text, return_offsets_mapping=True, truncation=False)
        full_ids = tokenized["input_ids"]
        full_offsets = tokenized["offset_mapping"]
        
        chunk_size = 1024
        stride = 256
        step = chunk_size - stride
        
        doc_preds = []
        doc_offsets = []
        
        # 2. Slide window across document
        for i in range(0, len(full_ids), step):
            chunk_ids = full_ids[i : i + chunk_size]
            chunk_offs = full_offsets[i : i + chunk_size]
            
            input_tensor = torch.tensor([chunk_ids]).to("cuda")
            
            # 3. Model Prediction with Class Penalty
            with torch.inference_mode():
                outputs = model(input_ids=input_tensor)
                logits = outputs.logits
                #logits[:, :, 0] -= 3.0  # Penalize 'O' class
                preds = torch.argmax(logits, dim=-1).cpu().numpy()[0]
                
            doc_preds.append(preds)
            doc_offsets.append(chunk_offs)
            
        # 4. Decode tokens back to exact character spans
        extracted_spans = decode_token_predictions_to_spans(raw_text, doc_preds, doc_offsets)
        predictions[doc_id] = extracted_spans

    except Exception as e:
        print(f"\nError processing document {doc_id}: {str(e)}")
        predictions[doc_id] = []

# 5. Save to Submission Format
os.makedirs("./phase2_gqa_lorma", exist_ok=True)
PREDICTION_PATH = "./phase2_gqa_lorma/prediction.json"
SUBMISSION_ZIP = "./phase2_gqa_lorma/submission.zip"

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print(f"\nFinal sanity check: Saved {len(predictions.keys())} document IDs to prediction.json")
print(f"L-NER Predictions complete! Submission saved to: {SUBMISSION_ZIP}")

Loading Test Data...


fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

Successfully loaded 35 documents from the test fold.


Generating L-NER Spans:   0%|          | 0/35 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(



Final sanity check: Saved 35 document IDs to prediction.json
L-NER Predictions complete! Submission saved to: ./phase2_gqa_lorma/submission.zip


In [12]:
# Run this in a new cell to diagnose boundary artifacts
first_doc_id = list(predictions.keys())[0]
raw_doc_text = test_data[first_doc_id]

print(f"--- Diagnostic: Top 20 Predictions for Doc {first_doc_id} ---")
for span in predictions[first_doc_id][:20]:
    start, end, label = span
    extracted_text = raw_doc_text[start:end]
    # Using repr() to expose hidden trailing spaces or newlines (e.g. 'Court \n')
    print(f"[{label: <10}] : {repr(extracted_text)}")

--- Diagnostic: Top 20 Predictions for Doc 1128835 ---
[CASENO    ] : 'COURT'
[DATE      ] : 'INDIA'
[DATE      ] : 'JURISDICTION'
[DATE      ] : 'APPEAL'
[DATE      ] : '6'
[DATE      ] : '2008'
[DATE      ] : 'RI'
[DATE      ] : 'OUT'
[DATE      ] : 'PETITION'
[CASENO    ] : '('
[DATE      ] : ')'
[DATE      ] : '15449 OF 2004'
[PREC      ] : 'AB'
[DATE      ] : 'LTD'
[CASENO    ] : 'ANR'
[DATE      ] : 'APPELLANT'
[CASENO    ] : 'STATE'
[PREC      ] : 'HARY'
[DATE      ] : 'ANA'
[CASENO    ] : 'ANR'
